In [1]:
import sys
import numpy as np

sys.path.append("../../../")
from Rain.Rain import Rain
sys.path.pop()

from keras.models import Sequential
from keras.layers import Dense, Activation, Dropout
import tensorflow as tf

2023-07-03 06:33:58.908131: I tensorflow/core/platform/cpu_feature_guard.cc:182] This TensorFlow binary is optimized to use available CPU instructions in performance-critical operations.
To enable the following instructions: AVX2 FMA, in other operations, rebuild TensorFlow with the appropriate compiler flags.
2023-07-03 06:33:59.695362: W tensorflow/compiler/tf2tensorrt/utils/py_utils.cc:38] TF-TRT Warning: Could not find TensorRT


In [2]:
import os

def clean():
    folder_paths = ["logs"]  # Replace with the folder path you want to delete files from
    file_extensions = [".npy", ".pkl", ".log"]  # Replace with the file extension you want to delete
    for folder_path in folder_paths:
        if os.path.exists(folder_path):
            for filename in os.listdir(folder_path):
                for file_extension in file_extensions:
                    if filename.endswith(file_extension):
                        file_path = os.path.join(folder_path, filename)
                        os.remove(file_path)
clean()

In [3]:
config = {
  "mode": "local",
  "partitions": 3,
  "num_of_workers": 3,
  "iterations": 3,
  "learning_type": "DL",
  "DL": {
    "lib": {
      "type": "tensorflow",
      "params": {
        "loss": tf.keras.losses.CategoricalCrossentropy(),
        "optimizer": tf.keras.optimizers.Adam(learning_rate=0.001),

      }
    },
    "lr": 0.001,
    "epochs": 2,
    "batch_size": 128,
  }
}

In [4]:
def get_train_data():
    return np.load("../../../data/MNIST/train_data.npy"), np.load(
        "../../../data/MNIST/train_labels.npy"
    )


def get_test_data():
    return np.load("../../../data/MNIST/test_data.npy"), np.load(
        "../../../data/MNIST/test_labels.npy"
    )


def partition_train_data(X_train, y_train, partitions):
    num_samples = X_train.shape[0]

    # Create an array of indices from 0 to num_samples - 1
    indices = np.arange(num_samples)

    # Shuffle the indices
    np.random.shuffle(indices)

    # Use the shuffled indices to shuffle the datasets
    X_train = X_train[indices]
    y_train = y_train[indices]

    X_train_partitions = []
    y_train_partitions = []

    partition_size = int(len(X_train) / partitions)

    for i in range(partitions):
        if i == partitions - 1:
            X_train_partitions.append(X_train[i * partition_size :])
            y_train_partitions.append(y_train[i * partition_size :])
        else:
            X_train_partitions.append(
                X_train[i * partition_size : (i + 1) * partition_size]
            )
            y_train_partitions.append(
                y_train[i * partition_size : (i + 1) * partition_size]
            )

    return X_train_partitions, y_train_partitions


In [5]:
def create_model():
    # network parameters
    hidden_units = 256
    dropout = 0.45
    input_size = 784
    num_labels = 10
    # model is a 3-layer MLP with ReLU and dropout after each layer
    model = Sequential()
    model.add(Dense(hidden_units, input_dim=input_size))
    model.add(Activation("relu"))
    model.add(Dropout(dropout))
    model.add(Dense(hidden_units))
    model.add(Activation("relu"))
    model.add(Dropout(dropout))
    model.add(Dense(num_labels))
    model.add(Activation("softmax"))
    return model

In [6]:
X_train, y_train = get_train_data()


In [7]:
model = create_model()
rain = Rain(config, model)

2023-07-03 06:34:00,704 [DEBUG] [Rain] Rain is initialized
2023-07-03 06:34:00,705 [DEBUG] [Provisioner] Creating coordinator
2023-07-03 06:34:00,706 [DEBUG] [TemporaryFilesManager] Creating temporary directory ../../../../Rain/data/coord/data/
2023-07-03 06:34:00,707 [DEBUG] [Coordinator Ambassador] Coordinator Ambassador is initialized
2023-07-03 06:34:00,708 [DEBUG] [Coordinator] Coordinator is initialized
2023-07-03 06:34:00,709 [DEBUG] [TemporaryFilesManager] Creating temporary directory ../../../../Rain/data/coord/data/
2023-07-03 06:34:00,710 [DEBUG] [LocalProvisioner] LocalProvisioner is initialized
2023-07-03 06:34:00,711 [DEBUG] [TemporaryFilesManager] Creating temporary directory ../../../../Rain/data/divider/data/
2023-07-03 06:34:00,713 [DEBUG] [TemporaryFilesManager] Creating temporary directory ../../../../Rain/data/divider/data/
2023-07-03 06:34:00,714 [DEBUG] [TemporaryFilesManager] Creating temporary directory ../../../../Rain/data/divider/data/


In [8]:
model = rain.train(X_train, y_train, strategy='async')

2023-07-03 06:34:00,719 [DEBUG] [Rain] Creating workers
2023-07-03 06:34:00,726 [INFO] [Provisioner] provisioner is serving
2023-07-03 06:34:00,727 [DEBUG] [Provisioner] Starting coordinator
2023-07-03 06:34:00,729 [INFO] [Coordinator] coordinator is serving
2023-07-03 06:34:00,729 [DEBUG] [Coordinator] sending the num of workers to the provisioner
2023-07-03 06:34:00,733 [DEBUG] [Provisioner] Received 'NumOfWorkers: 3
' from the coordinator to define the number of workers
2023-07-03 06:34:00,735 [DEBUG] [Coordinator] sent Success receiving the number of workers to the provisioner
2023-07-03 06:34:00,736 [DEBUG] [LocalProvisioner] Creating 3 workers
2023-07-03 06:34:00,737 [DEBUG] [TemporaryFilesManager] Creating temporary directory ../../../../Rain/data/worker/data/
2023-07-03 06:34:00,738 [INFO] [Worker_50151] Worker is running on port: 50151
2023-07-03 06:34:00,739 [DEBUG] [TemporaryFilesManager] Creating temporary directory ../../../../Rain/data/worker/data/
2023-07-03 06:34:00,740

Epoch 1/2
Epoch 1/2
Epoch 1/2
157/157 [==============================] - 1s 5ms/step - loss: 0.6941 - accuracy: 0.7833
Epoch 2/2
157/157 [==============================] - 1s 5ms/step - loss: 0.6952 - accuracy: 0.7815
Epoch 2/2
157/157 [==============================] - 1s 5ms/step - loss: 0.6996 - accuracy: 0.7789
Epoch 2/2
157/157 [==============================] - 1s 5ms/step - loss: 0.3004 - accuracy: 0.9084
sending data to coordinator
157/157 [==============================] - 1s 5ms/step - loss: 0.3016 - accuracy: 0.9091
sending data to coordinator
157/157 [==============================] - 1s 5ms/step - loss: 0.3028 - accuracy: 0.9101
sending data to coordinator


2023-07-03 06:34:34,013 [DEBUG] [DividerAmbassador] divider received: Executed! after executing the model on worker 3
2023-07-03 06:34:34,014 [DEBUG] [DividerAmbassador] divider begins downloading ../../../../Rain/data/divider/data/3_3_trained.pkl from worker3
2023-07-03 06:34:34,022 [DEBUG] [DividerAmbassador] divider received: Executed! after executing the model on worker 2
2023-07-03 06:34:34,023 [DEBUG] [DividerAmbassador] divider begins downloading ../../../../Rain/data/divider/data/2_2_trained.pkl from worker2
2023-07-03 06:34:34,043 [DEBUG] [DividerAmbassador] divider received: Executed! after executing the model on worker 1
2023-07-03 06:34:34,045 [DEBUG] [DividerAmbassador] divider begins downloading ../../../../Rain/data/divider/data/1_1_trained.pkl from worker1
2023-07-03 06:34:34,259 [DEBUG] [DividerAmbassador] Downloaded ../../../../Rain/data/divider/data/3_3_trained.pkl from worker3 successfully
2023-07-03 06:34:34,277 [DEBUG] [DeepLearning] Asynchronous update is done by

Epoch 1/2
Epoch 1/2
Epoch 1/2
157/157 [==============================] - 1s 5ms/step - loss: 0.2581 - accuracy: 0.9233
Epoch 2/2
157/157 [==============================] - 1s 5ms/step - loss: 0.3020 - accuracy: 0.9115
Epoch 2/2
Epoch 2/2
157/157 [==============================] - 1s 5ms/step - loss: 0.2464 - accuracy: 0.9259
sending data to coordinator
sending data to coordinator


2023-07-03 06:34:40,182 [DEBUG] [DividerAmbassador] divider received: Executed! after executing the model on worker 3
2023-07-03 06:34:40,183 [DEBUG] [DividerAmbassador] divider begins downloading ../../../../Rain/data/divider/data/3_3_trained.pkl from worker3
2023-07-03 06:34:40,189 [DEBUG] [DividerAmbassador] divider received: Executed! after executing the model on worker 2
2023-07-03 06:34:40,190 [DEBUG] [DividerAmbassador] divider begins downloading ../../../../Rain/data/divider/data/2_2_trained.pkl from worker2
2023-07-03 06:34:40,195 [DEBUG] [DividerAmbassador] divider received: Executed! after executing the model on worker 1
2023-07-03 06:34:40,196 [DEBUG] [DividerAmbassador] divider begins downloading ../../../../Rain/data/divider/data/1_1_trained.pkl from worker1
2023-07-03 06:34:40,417 [DEBUG] [DividerAmbassador] Downloaded ../../../../Rain/data/divider/data/3_3_trained.pkl from worker3 successfully
2023-07-03 06:34:40,426 [DEBUG] [DeepLearning] Asynchronous update is done by

Epoch 1/2
Epoch 1/2
Epoch 1/2
157/157 [==============================] - 1s 5ms/step - loss: 0.1963 - accuracy: 0.9425
Epoch 2/2
Epoch 2/2
157/157 [==============================] - 1s 5ms/step - loss: 0.2236 - accuracy: 0.9328
Epoch 2/2
157/157 [==============================] - 1s 5ms/step - loss: 0.1612 - accuracy: 0.9531
sending data to coordinator
157/157 [==============================] - 1s 5ms/step - loss: 0.1772 - accuracy: 0.9482
sending data to coordinator


2023-07-03 06:34:46,432 [DEBUG] [DividerAmbassador] divider received: Executed! after executing the model on worker 1
2023-07-03 06:34:46,433 [DEBUG] [DividerAmbassador] divider begins downloading ../../../../Rain/data/divider/data/1_1_trained.pkl from worker1
2023-07-03 06:34:46,443 [DEBUG] [DividerAmbassador] divider received: Executed! after executing the model on worker 3
2023-07-03 06:34:46,444 [DEBUG] [DividerAmbassador] divider begins downloading ../../../../Rain/data/divider/data/3_3_trained.pkl from worker3
2023-07-03 06:34:46,573 [DEBUG] [DividerAmbassador] Downloaded ../../../../Rain/data/divider/data/1_1_trained.pkl from worker1 successfully
2023-07-03 06:34:46,580 [DEBUG] [DeepLearning] Asynchronous update is done by worker 1
2023-07-03 06:34:46,598 [DEBUG] [DividerAmbassador] Downloaded ../../../../Rain/data/divider/data/3_3_trained.pkl from worker3 successfully
2023-07-03 06:34:46,601 [DEBUG] [DeepLearning] Iteration 3/3 complete for worker 1.
2023-07-03 06:34:46,605 [DE

sending data to coordinator


2023-07-03 06:34:47,396 [DEBUG] [DividerAmbassador] divider received: Executed! after executing the model on worker 2
2023-07-03 06:34:47,397 [DEBUG] [DividerAmbassador] divider begins downloading ../../../../Rain/data/divider/data/2_2_trained.pkl from worker2
2023-07-03 06:34:47,472 [DEBUG] [DividerAmbassador] Downloaded ../../../../Rain/data/divider/data/2_2_trained.pkl from worker2 successfully
2023-07-03 06:34:47,478 [DEBUG] [DeepLearning] Asynchronous update is done by worker 2
2023-07-03 06:34:47,495 [DEBUG] [DeepLearning] Iteration 3/3 complete for worker 2.
2023-07-03 06:34:47,497 [DEBUG] [DividerAmbassador] divider ambassador stopped serving
2023-07-03 06:34:47,497 [DEBUG] [Divider] Divider stopped serving
2023-07-03 06:34:47,498 [DEBUG] [DividerAmbassador] divider ambassador stopped serving
2023-07-03 06:34:47,498 [DEBUG] [Divider] Divider stopped serving
2023-07-03 06:34:47,499 [INFO] [Provisioner] provisioner stopped serving


In [9]:
X_test, y_test = get_test_data()
loss, acc = model.evaluate(X_test, y_test, batch_size=config["DL"]["batch_size"])
print("\nTest accuracy: %.1f%%" % (100.0 * acc))

79/79 [==============================] - 0s 1ms/step - loss: 0.1024 - accuracy: 0.9684

Test accuracy: 96.8%


In [10]:
model = rain.train(X_train, y_train, strategy='sync')

2023-07-03 06:34:47,861 [DEBUG] [Rain] Creating workers
2023-07-03 06:34:47,863 [INFO] [Provisioner] provisioner is serving
2023-07-03 06:34:47,865 [DEBUG] [Provisioner] Starting coordinator
2023-07-03 06:34:47,866 [INFO] [Coordinator] coordinator is serving
2023-07-03 06:34:47,867 [DEBUG] [Coordinator] sending the num of workers to the provisioner
2023-07-03 06:34:47,869 [DEBUG] [Provisioner] Received 'NumOfWorkers: 3
' from the coordinator to define the number of workers
2023-07-03 06:34:47,871 [DEBUG] [Coordinator] sent Success receiving the number of workers to the provisioner
2023-07-03 06:34:47,871 [DEBUG] [LocalProvisioner] Creating 3 workers
2023-07-03 06:34:47,872 [DEBUG] [TemporaryFilesManager] Creating temporary directory ../../../../Rain/data/worker/data/
2023-07-03 06:34:47,874 [INFO] [Worker_50151] Worker is running on port: 50151
2023-07-03 06:34:47,874 [INFO] [Worker_50151] Worker is running on port: 50151
2023-07-03 06:34:47,876 [DEBUG] [TemporaryFilesManager] Creating

Epoch 1/2
Epoch 1/2
Epoch 1/2
157/157 [==============================] - 2s 8ms/step - loss: 0.1557 - accuracy: 0.9536
Epoch 2/2
157/157 [==============================] - 2s 9ms/step - loss: 0.1558 - accuracy: 0.9529
Epoch 2/2
157/157 [==============================] - 2s 10ms/step - loss: 0.1603 - accuracy: 0.9524
Epoch 2/2
157/157 [==============================] - 2s 10ms/step - loss: 0.1321 - accuracy: 0.9598
sending data to coordinator
sending data to coordinator
sending data to coordinator


2023-07-03 06:35:24,890 [DEBUG] [DividerAmbassador] divider received: Executed! after executing the model on worker 2
2023-07-03 06:35:24,891 [DEBUG] [DividerAmbassador] divider begins downloading ../../../../Rain/data/divider/data/2_1_trained.pkl from worker2
2023-07-03 06:35:24,951 [DEBUG] [DividerAmbassador] divider received: Executed! after executing the model on worker 3
2023-07-03 06:35:24,952 [DEBUG] [DividerAmbassador] divider begins downloading ../../../../Rain/data/divider/data/3_1_trained.pkl from worker3
2023-07-03 06:35:24,962 [DEBUG] [DividerAmbassador] divider received: Executed! after executing the model on worker 1
2023-07-03 06:35:24,964 [DEBUG] [DividerAmbassador] divider begins downloading ../../../../Rain/data/divider/data/1_1_trained.pkl from worker1
2023-07-03 06:35:25,018 [DEBUG] [DividerAmbassador] Downloaded ../../../../Rain/data/divider/data/2_1_trained.pkl from worker2 successfully
2023-07-03 06:35:25,136 [DEBUG] [DividerAmbassador] Downloaded ../../../../Ra

Epoch 1/2
Epoch 1/2
Epoch 1/2
157/157 [==============================] - 1s 5ms/step - loss: 0.1333 - accuracy: 0.9602
Epoch 2/2
157/157 [==============================] - 1s 5ms/step - loss: 0.1364 - accuracy: 0.9590
Epoch 2/2
157/157 [==============================] - 1s 6ms/step - loss: 0.1158 - accuracy: 0.9646
sending data to coordinator
157/157 [==============================] - 1s 6ms/step - loss: 0.1231 - accuracy: 0.9632
sending data to coordinator
157/157 [==============================] - 1s 5ms/step - loss: 0.1214 - accuracy: 0.9632
sending data to coordinator


2023-07-03 06:35:31,691 [DEBUG] [DividerAmbassador] divider received: Executed! after executing the model on worker 2
2023-07-03 06:35:31,693 [DEBUG] [DividerAmbassador] divider begins downloading ../../../../Rain/data/divider/data/2_2_trained.pkl from worker2
2023-07-03 06:35:31,721 [DEBUG] [DividerAmbassador] divider received: Executed! after executing the model on worker 1
2023-07-03 06:35:31,722 [DEBUG] [DividerAmbassador] divider begins downloading ../../../../Rain/data/divider/data/1_2_trained.pkl from worker1
2023-07-03 06:35:31,739 [DEBUG] [DividerAmbassador] divider received: Executed! after executing the model on worker 3
2023-07-03 06:35:31,740 [DEBUG] [DividerAmbassador] divider begins downloading ../../../../Rain/data/divider/data/3_2_trained.pkl from worker3
2023-07-03 06:35:31,912 [DEBUG] [DividerAmbassador] Downloaded ../../../../Rain/data/divider/data/2_2_trained.pkl from worker2 successfully
2023-07-03 06:35:31,972 [DEBUG] [DividerAmbassador] Downloaded ../../../../Ra

Epoch 1/2
Epoch 1/2
Epoch 1/2
157/157 [==============================] - 1s 5ms/step - loss: 0.1183 - accuracy: 0.9635
Epoch 2/2
157/157 [==============================] - 1s 5ms/step - loss: 0.1211 - accuracy: 0.9638
Epoch 2/2
157/157 [==============================] - 1s 5ms/step - loss: 0.1030 - accuracy: 0.9668
sending data to coordinator
157/157 [==============================] - 1s 5ms/step - loss: 0.1049 - accuracy: 0.9676
sending data to coordinator


2023-07-03 06:35:38,027 [DEBUG] [DividerAmbassador] divider received: Executed! after executing the model on worker 3
2023-07-03 06:35:38,028 [DEBUG] [DividerAmbassador] divider begins downloading ../../../../Rain/data/divider/data/3_3_trained.pkl from worker3
2023-07-03 06:35:38,100 [DEBUG] [DividerAmbassador] divider received: Executed! after executing the model on worker 2
2023-07-03 06:35:38,101 [DEBUG] [DividerAmbassador] divider begins downloading ../../../../Rain/data/divider/data/2_3_trained.pkl from worker2
2023-07-03 06:35:38,133 [DEBUG] [DividerAmbassador] Downloaded ../../../../Rain/data/divider/data/3_3_trained.pkl from worker3 successfully
2023-07-03 06:35:38,201 [DEBUG] [DividerAmbassador] divider received: Executed! after executing the model on worker 1
2023-07-03 06:35:38,202 [DEBUG] [DividerAmbassador] divider begins downloading ../../../../Rain/data/divider/data/1_3_trained.pkl from worker1
2023-07-03 06:35:38,208 [DEBUG] [DividerAmbassador] Downloaded ../../../../Ra

In [11]:
X_test, y_test = get_test_data()
loss, acc = model.evaluate(X_test, y_test, batch_size=config["DL"]["batch_size"])
print("\nTest accuracy: %.1f%%" % (100.0 * acc))

79/79 [==============================] - 0s 1ms/step - loss: 0.0796 - accuracy: 0.9767

Test accuracy: 97.7%
